# Download GOES (WFABBA), MODIS, and VIIRS Heat Detection Data from WIFIRE Firemap
### What this notebook does:

1. Search for GOES, MODIS, and VIIRS heat detections by datetime range
2. Download all detection features from the WIFIRE Firemap Geoserver
3. Visualize heat detections on a Folium map
4. Save as JSON


### Data source:

* WIFIRE Firemap Geoserver
* Layers: 
	- ssec_wfabba_goes_no_low_code
	- view_modis_c6
	- view_viirs_iband
* Contains actual mapped detections with timestamps

### Requirements:

* Internet connection
* Fire name
* Fire start/stop datetimes

# Dependencies

In [ ]:
import requests
import json
import sys
from datetime import datetime, timezone
import folium
import os

# Fire Data Collection


Testing the Geoserver API Query

In [ ]:
# ========== USER INPUTS ==========

fire_name = "Border 2"  # Your chosen fire name will be used for download's filename

# Bounding Box
	# Syntax: "LowerLeft_Longitude,LowerLeft_Latitude,UpperRight_Longitude,UpperRight_Latitude,'EPSG:Code'"
	# For a complete CA BBOX, enter: "-124.5,32.45,-114.0,42.10,'EPSG:4326'"
LL_lon = -116.93191
LL_lat = 32.56482
UR_lon = -116.81826
UR_lat = 32.67451
bbox = f"{LL_lon},{LL_lat},{UR_lon},{UR_lat},'EPSG:4326'"
map_center = [(LL_lat+UR_lat)/2.0, (LL_lon+UR_lon)/2.0]

# Specify a datetime & epoch (yyyy, m, d, h, m, s)
dt_start = datetime(2025, 1, 22, 0, 0, 1)    # Note: the first available GOES detection starts from (2017, 5, 20, 21, 35, 0)
dt_end = datetime(2025, 2, 1, 23, 59, 59)   # Note: Large datetime ranges (ex: more than 1 year) may overtax your computer's resources

# ==================================

GOES Query

In [ ]:
# Convert datetime to UTC
dt_start_utc = dt_start.replace(tzinfo=timezone.utc).isoformat(sep='T').replace('+00:00', 'z')
dt_end_utc = dt_end.replace(tzinfo=timezone.utc).isoformat(sep='T').replace('+00:00', 'z')

# GeoServer WFS URL configuration
goes_url = "https://sdge.sdsc.edu/geoserver/wfs"
goes_params = {
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'maxFeatures': '100000',
    'exceptions': 'application/json',
    'typeName': 'WIFIRE:ssec_wfabba_goes_no_low_code',
    'CQL_FILTER': f"data_time BETWEEN '{dt_start_utc}' AND '{dt_end_utc}' AND BBOX(geom,{bbox})",
    'outputFormat': 'application/json'
}

# Request the data
goes_r = requests.get(goes_url, params=goes_params, timeout=60)
if goes_r.status_code != 200:
    print(r.text)
    sys.exit(1)

goes_j = goes_r.json()
goes_f = goes_j.get("features", [])

if not goes_f:
    raise ValueError(
        f"No GOES dections found for '{fire_name}'.\n"
        f"Check that the datetime range is valid.\n"
        f"Hint: the first available GOES detection starts from 2016-08-11T09:20:00Z"
    )

print(f"✓ Found {len(goes_f)} detections \n")

# View the JSON
print(json.dumps(goes_j,indent=2))   # WARNING: Large datetime ranges may overtax your computer's resources

VIIRS & MODIS Query

In [ ]:
# Convert datetime to epoch
epoch_start = int(dt_start.timestamp())
epoch_end = int(dt_end.timestamp())

# GeoServer WFS URL configuration
viirs_modis_url = "https://firemap.sdsc.edu/geoserver/wfs"
viirs_modis_params = {
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'maxFeatures': '100000',
    'exceptions': 'application/json',
    'typeName': 'WIFIRE:view_modis_c6,WIFIRE:view_viirs_iband',
    'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
    'outputFormat': 'application/json'
}

# Request the data
viirs_modis_r = requests.get(viirs_modis_url, params=viirs_modis_params, timeout=60)
if viirs_modis_r.status_code != 200:
    print(viirs_modis_r.text)
    sys.exit(1)

viirs_modis_j = viirs_modis_r.json()
viirs_modis_f = viirs_modis_j.get("features", [])

if not viirs_modis_f:
    raise ValueError(
        f"No MODIS dections found for '{fire_name}'.\n"
        f"Check that the datetime range is valid.\n"
        f"Hint: the first available MODIS detection starts from 2016-08-11T09:20:00Z"
    )

print(f"✓ Found {len(viirs_modis_f)} detections \n")

# View the JSON
print(json.dumps(viirs_modis_j,indent=2))   # WARNING: Large datetime ranges may overtax your computer's resources

In [ ]:
# # For full Traceback 
# %tb

Previewing the JSON response in GIS

In [ ]:
# Create map object
m = folium.Map(location=map_center, tiles='cartodbpositron', zoom_start=11)

#GOES
# Add 1000m Radius Circle Marker for GOES GeoJson features
goes_features = folium.GeoJson(goes_j,
    name="GOES",
	marker=folium.Circle(
        radius=1000, # Radius in meters
		color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.01),
	tooltip = folium.GeoJsonTooltip(
		fields=['id', 'satellite', 'data_time', 'frp'])
).add_to(m)

# MODIS
# Filter for objects where 'id' contains the partial string 'view_modis'
search_modis = "view_modis"
modis_subset = [feat for feat in viirs_modis_j['features'] if search_modis in feat['id']]
# 3. Create new GeoJSON structure
modis_geojson = {
    "type": "FeatureCollection",
    "features": modis_subset}
# Add 500m Radius Circle Marker for MODIS GeoJson features
modis_features = folium.features.GeoJson(modis_geojson,
	name="MODIS",
    marker=folium.Circle(
        radius=500, # Radius in meters
		color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.2),
	tooltip = folium.GeoJsonTooltip(
		fields=['satellite', 'acq_date', 'confidence', 'frp', 'daynight'])
).add_to(m)

# VIIRS
# Filter for objects where 'id' contains the partial string 'view_viirs'
search_viirs = "view_viirs"
viirs_subset = [feat for feat in viirs_modis_j['features'] if search_viirs in feat['id']]
# 3. Create new GeoJSON structure
viirs_geojson = {
    "type": "FeatureCollection",
    "features": viirs_subset}
# Add 375m Radius Circle Marker for VIIRS GeoJson features
viirs_features = folium.GeoJson(viirs_geojson,
    name="VIIRS",
	marker=folium.Circle(
        radius=375, # Radius in meters
		color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.2),
	tooltip = folium.GeoJsonTooltip(
		fields=['satellite', 'acq_date', 'confidence', 'frp', 'daynight'])
).add_to(m)

# Add Layer Control to Toggle Layers
folium.LayerControl().add_to(m)

# Preview map
m   # WARNING: Large datetime ranges may overtax your computer's resources


Create Output Filename & Directory

In [ ]:
# # Set Output folder & filename prefix
# filename = fire_name.replace(" ", "_")
# os.makedirs(f"./Fires/{filename}", exist_ok=True)

# print(filename)

Getting GOES JSON

In [ ]:
# # GeoServer WFS URL JSON configuration
# url = "https://sdge.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:ssec_wfabba_goes_no_low_code',
#     'CQL_FILTER': f"data_time BETWEEN '{dt_start_utc}' AND '{dt_end_utc}' AND BBOX(geom,{bbox})",
#     'outputFormat': 'application/json'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     j = r.json()    
#     with open(f"./Fires/{filename}/{filename}_goes.json", "w", encoding="utf-8") as file:
#         json.dump(j, file)
#     print("GOES JSON saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)

Getting MODIS JSON

In [ ]:
# # GeoServer WFS URL JSON configuration
# url = "https://firemap.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:view_modis_c6',
#     'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
#     'outputFormat': 'application/json'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     j = r.json()    
#     with open(f"./Fires/{filename}/{filename}_modis.json", "w", encoding="utf-8") as file:
#         json.dump(j, file)
#     print("MODIS JSON saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)

Getting VIIRS JSON

In [ ]:
# # GeoServer WFS URL JSON configuration
# url = "https://firemap.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:view_viirs_iband',
#     'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
#     'outputFormat': 'application/json'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     j = r.json()    
#     with open(f"./Fires/{filename}/{filename}_viirs.json", "w", encoding="utf-8") as file:
#         json.dump(j, file)
#     print("VIIRS JSON saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)